# Phase 1: Continued Pre-training on Medical Literature (GPU)

**Train Qwen2.5-7B on 14 medical books using LoRA on RTX 5090**

- **Duration:** 12-14 GPU hours (RTX 5090)
- **Memory:** ~20-24 GB VRAM
- **Output:** Medical-grounded Qwen model ready for Phase 2

## 1. Setup and Environment

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

import torch
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("PHASE 1: CONTINUED PRE-TRAINING (GPU)")
print("="*80)
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
print("="*80)
print()

## 2. Verify Data

In [ ]:
DATA_FILE = r'C:\Users\Krish\Downloads\LLM_Finetuning\full_medical_data.txt'
OUTPUT_DIR = r'C:\Users\Krish\Downloads\LLM_Finetuning\qwen_medical_pretrained_gpu'
LORA_OUTPUT = r'C:\Users\Krish\Downloads\LLM_Finetuning\qwen_medical_lora_gpu'
CACHE_DIR = r'C:\Users\Krish\Downloads\LLM_Finetuning\.cache'

Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)
Path(LORA_OUTPUT).mkdir(exist_ok=True, parents=True)
Path(CACHE_DIR).mkdir(exist_ok=True, parents=True)

print("[1/6] Verifying data file...")

if not os.path.exists(DATA_FILE):
    print(f"ERROR: Data file not found: {DATA_FILE}")
    print(f"Run: extract_pdf_data.py")
else:
    data_size_mb = os.path.getsize(DATA_FILE) / (1024 * 1024)
    with open(DATA_FILE, 'r', encoding='utf-8') as f:
        text = f.read()
    word_count = len(text.split())
    
    print(f"  ✓ Data file verified")
    print(f"  Path: {DATA_FILE}")
    print(f"  Size: {data_size_mb:.2f} MB")
    print(f"  Words: {word_count:,}")
    print()

## 3. Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B"

print("[2/6] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"  ✓ Tokenizer loaded")
print(f"  Model: {MODEL_NAME}")
print(f"  Vocab size: {len(tokenizer)}")
print()

## 4. Load Base Model

In [ ]:
from transformers import AutoModelForCausalLM

print("[3/6] Loading base model...")
print("  This may take 2-5 minutes...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # GPU: use float16 for speed
    device_map="auto",          # Automatic GPU/CPU allocation
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)

print(f"  ✓ Model loaded")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
print(f"  Device: {next(model.parameters()).device}")
print()

## 5. Configure LoRA

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

print("[4/6] Configuring LoRA...")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                          # Rank
    lora_alpha=32,                 # Alpha (32/16 = 2.0)
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"  ✓ LoRA configured")
print(f"  LoRA rank: {lora_config.r}")
print(f"  LoRA alpha: {lora_config.lora_alpha}")
print(f"  Trainable params: {trainable_params / 1e6:.1f}M ({100 * trainable_params / all_params:.2f}%)")
print(f"  Total params: {all_params / 1e9:.1f}B")
print()

model.print_trainable_parameters()

## 6. Prepare Dataset

In [ ]:
from transformers import TextDataset, DataCollatorForLanguageModeling

print("[5/6] Preparing dataset...")

train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path=DATA_FILE,
    block_size=512,  # Context window
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM
)

print(f"  ✓ Dataset prepared")
print(f"  Dataset size: {len(train_dataset)} blocks")
print(f"  Block size: 512 tokens")
print()

## 7. Configure Training (GPU Optimized)

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

print("[6/6] Configuring training...")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=2,              # 2 epochs
    per_device_train_batch_size=2,   # RTX 5090: batch size 2
    gradient_accumulation_steps=8,   # Effective batch: 16
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=50,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=True,                      # GPU: float16
    gradient_checkpointing=True,    # Save memory
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    log_level="info",
    report_to=["tensorboard"],
    push_to_hub=False,
)

print(f"  ✓ Training configuration ready")
print(f"  Batch size (effective): {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Mixed precision: float16")
print(f"  Estimated time: 12-14 GPU hours")
print()
print("Starting training...")
print(f"Time: {datetime.now().isoformat()}")
print()

## 8. Train Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

train_result = trainer.train()

print()
print("="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Time: {datetime.now().isoformat()}")
print()

## 9. Save LoRA Adapters

In [ ]:
print("Saving LoRA adapters...")

model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)

config = {
    'model': MODEL_NAME,
    'device': 'GPU (RTX 5090)',
    'data_file': DATA_FILE,
    'num_words': 899042,
    'lora_rank': lora_config.r,
    'lora_alpha': lora_config.lora_alpha,
    'training_epochs': training_args.num_train_epochs,
    'training_loss': float(train_result.training_loss),
    'timestamp': datetime.now().isoformat(),
}

with open(os.path.join(LORA_OUTPUT, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ LoRA saved to: {LORA_OUTPUT}")
print(f"✓ Config saved")
print()
print("="*80)
print("PHASE 1 COMPLETE")
print("="*80)
print()
print("Next: Run Phase 2 - Instruction Fine-tuning")
print("  Run: Phase2_Instruction_Finetuning_GPU.ipynb")

## Notes

- **Total time:** 12-14 GPU hours on RTX 5090
- **Checkpoints:** Saved every 300 steps in `qwen_medical_pretrained_gpu/`
- **Resume:** If interrupted, run training cell again to resume from last checkpoint
- **Memory:** ~20-24 GB VRAM usage

If running out of memory:
- Reduce `per_device_train_batch_size` from 2 to 1
- Reduce `block_size` from 512 to 256